# Spendly model development V2

This notebook implements the fixes discovered by the independent generalization test:

- forecasting infers recurring-payment schedules and amounts from each user's past transactions;
- full weeks only are used, preventing partial-boundary distortion;
- the forecast target is normalized by income and models long-tailed weekly spending robustly;
- anomaly rarity rules exclude normal recurring categories;
- merchant-hopping, split-transaction and user-relative nighttime features are added.

The 2025 stress dataset is now **development evidence**, not a final test. V2 is frozen here and
must next be evaluated once on a newly generated untouched seed. No app files are changed.

## How to run

1. Open in Colab and run all cells on a CPU runtime.
2. Upload `combined_finance_raw.csv` and `spendly_generalization_output.zip` when prompted.
3. Download `spendly_development_v2_output.zip`.
4. Paste the block between `SPENDLY_DEVELOPMENT_V2_RESULT_START` and
   `SPENDLY_DEVELOPMENT_V2_RESULT_END` back into the task.

Expected runtime is approximately 4–7 minutes.

In [ ]:
import importlib.util
import subprocess
import sys

required = {
    "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "joblib": "joblib",
}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependency check: PASS")

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import time
import warnings
import zipfile
from collections import defaultdict, deque
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor, IsolationForest
from sklearn.metrics import (
    confusion_matrix, mean_squared_error, precision_recall_curve,
    precision_recall_fscore_support, r2_score,
)
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
SEED = 42
LOOKBACK = 8
RECURRING_CATEGORIES = ["Rent", "Utilities", "Subscriptions", "Savings"]
KNOWN_RECURRING = set(RECURRING_CATEGORIES)
OUTPUT_DIR = Path("spendly_development_v2_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
PRODUCTION_TARGETS = {
    "forecast_wape_max": 30.0, "forecast_r2_min": 0.50,
    "anomaly_precision_min": 0.60, "anomaly_recall_min": 0.75,
}
print({"seed": SEED, "lookback_weeks": LOOKBACK, "targets": PRODUCTION_TARGETS})

In [ ]:
original_path = Path(os.environ.get("SPENDLY_DATA_PATH", "combined_finance_raw.csv"))
bundle_path = Path(os.environ.get("SPENDLY_GENERALIZATION_ZIP", "spendly_generalization_output.zip"))
if not original_path.exists():
    matches = list(Path.cwd().rglob("combined_finance_raw.csv"))
    if matches:
        original_path = matches[0]
if not bundle_path.exists():
    matches = list(Path.cwd().rglob("spendly_generalization_output.zip"))
    if matches:
        bundle_path = matches[0]
if not original_path.exists() or not bundle_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        for filename in uploaded:
            candidate = Path(filename)
            if candidate.name == "combined_finance_raw.csv":
                original_path = candidate
            elif candidate.suffix.lower() == ".zip":
                bundle_path = candidate
    except Exception as exc:
        raise FileNotFoundError(
            "Provide combined_finance_raw.csv and spendly_generalization_output.zip."
        ) from exc
assert original_path.exists() and bundle_path.exists()

original_sha256 = hashlib.sha256(original_path.read_bytes()).hexdigest()
expected_original_sha256 = "b39cb5e5725a504c8880358edd160945323e208625af441a8680decd0f34b834"
assert original_sha256 == expected_original_sha256, "Original training CSV hash does not match"
original = pd.read_csv(original_path, low_memory=False)
with zipfile.ZipFile(bundle_path) as archive:
    member = "spendly_generalization_outputs/independent_stress_finance.csv"
    assert member in archive.namelist(), "Stress CSV is missing from the evidence bundle"
    with archive.open(member) as handle:
        stress = pd.read_csv(handle, low_memory=False)
for frame in [original, stress]:
    frame["transaction_timestamp"] = pd.to_datetime(frame["transaction_timestamp"], errors="raise")
    frame["amount"] = pd.to_numeric(frame["amount"], errors="raise")
    frame["is_recurring"] = frame["is_recurring"].fillna(False).astype(str).str.lower().isin(
        {"true", "1", "yes", "y", "t"}
    )
    frame["transaction_type"] = frame["transaction_type"].astype(str).str.lower()
    frame["category"] = frame["category"].astype(str).str.strip()
    assert frame["transaction_id"].is_unique
assert set(original["user_id"]).isdisjoint(set(stress["user_id"]))
input_summary = pd.DataFrame([
    {"dataset": "original", "rows": len(original), "users": original["user_id"].nunique(),
     "start": original["transaction_timestamp"].min(), "end": original["transaction_timestamp"].max()},
    {"dataset": "stress_development", "rows": len(stress), "users": stress["user_id"].nunique(),
     "start": stress["transaction_timestamp"].min(), "end": stress["transaction_timestamp"].max()},
])
display(input_summary)
print("Input identity, schema and unseen-user checks: PASS")

## 1. Dynamic recurring-schedule forecasting

Recurring due dates are inferred from preceding transactions only. A target week receives an
expected recurring amount only when the inferred next monthly occurrence falls inside that full
Monday–Sunday week. The model then predicts weekly spending as a fraction of income, improving
stability across income levels and unseen users.

In [ ]:
def bools(series: pd.Series) -> pd.Series:
    return series.fillna(False).astype(str).str.lower().isin({"true", "1", "yes", "y", "t"})


def full_week_bounds(data: pd.DataFrame) -> tuple[pd.Timestamp, pd.Timestamp]:
    min_day = data["transaction_timestamp"].min().normalize()
    max_day = data["transaction_timestamp"].max().normalize()
    first_period = min_day.to_period("W-SUN").start_time
    last_period = max_day.to_period("W-SUN").start_time
    first_full = first_period if first_period >= min_day else first_period + pd.Timedelta(days=7)
    last_full = last_period if last_period + pd.Timedelta(days=6) <= max_day else last_period - pd.Timedelta(days=7)
    return first_full, last_full


def next_due_features(times: pd.DatetimeIndex, amounts: np.ndarray, target_start: pd.Timestamp, target_end: pd.Timestamp, income: float) -> list[float]:
    if len(times) == 0:
        return [0.0, 0.0, 0.0, 1.0, 0.0, 0.0]
    times = pd.DatetimeIndex(times[-8:])
    amounts = np.asarray(amounts[-8:], dtype=float)
    n = len(times)
    expected_amount = float(np.median(amounts[-3:]))
    if n >= 2:
        intervals = np.diff(times.asi8) / 86_400_000_000_000.0
        recent_intervals = intervals[-4:]
        regularity = float(np.exp(-np.std(recent_intervals) / 12.0))
        next_due = times[-1] + pd.DateOffset(months=1)
    else:
        regularity = 0.35
        next_due = times[-1] + pd.DateOffset(months=1)
    while next_due < target_start:
        next_due = next_due + pd.DateOffset(months=1)
    due_flag = float((next_due >= target_start) and (next_due <= target_end))
    month_span = max(1, (times[-1].year - times[0].year) * 12 + times[-1].month - times[0].month + 1)
    payment_rate = min(1.0, n / month_span)
    confidence = min(1.0, n / 3.0) * regularity * payment_rate
    amount_ratio = expected_amount / max(income, 1.0)
    days_since = min(90.0, max(0.0, (target_start - times[-1]).total_seconds() / 86400.0)) / 90.0
    next_day_angle = 2 * math.pi * (next_due.day - 1) / 31
    return [
        due_flag, amount_ratio, confidence, days_since,
        math.sin(next_day_angle), math.cos(next_day_angle),
    ]


def build_samples(data: pd.DataFrame, source: str) -> dict[str, np.ndarray]:
    expense = data[data["transaction_type"].eq("expense")].copy()
    expense["period_start"] = expense["transaction_timestamp"].dt.to_period("W-SUN").dt.start_time
    expense["nonrec_amount"] = np.where(~expense["is_recurring"], expense["amount"], 0.0)
    expense["rec_amount"] = np.where(expense["is_recurring"], expense["amount"], 0.0)
    first_full, last_full = full_week_bounds(data)
    categories = sorted(set(data["category"]) - {"Income"})
    expected_categories = [
        "Airtime and Data", "Education", "Entertainment", "Food", "Healthcare",
        "Irregular Expenses", "Rent", "Savings", "Subscriptions", "Transport", "Utilities",
    ]
    assert categories == expected_categories
    profile_cols = ["user_id", "user_age", "user_monthly_income", "user_employment_status", "user_income_day"]
    profiles = data[profile_cols].drop_duplicates("user_id").set_index("user_id")
    rows = []
    weekly_feature_names = [
        "total_spending", "nonrec_spending", "recurring_spending", "transaction_count",
        "average_amount", *["category_" + c.lower().replace(" ", "_") for c in categories],
    ]
    for user_id, user_expense in expense.groupby("user_id", sort=True):
        weeks = pd.date_range(first_full, last_full, freq="W-MON")
        base = user_expense.groupby("period_start").agg(
            total_spending=("amount", "sum"), nonrec_spending=("nonrec_amount", "sum"),
            recurring_spending=("rec_amount", "sum"), transaction_count=("transaction_id", "size"),
            average_amount=("amount", "mean"),
        )
        cat = user_expense.pivot_table(index="period_start", columns="category", values="amount", aggfunc="sum", fill_value=0.0)
        weekly = pd.DataFrame(index=weeks).join(base).join(cat).fillna(0.0)
        for category in categories:
            if category not in weekly:
                weekly[category] = 0.0
        weekly = weekly.rename(columns={c: "category_" + c.lower().replace(" ", "_") for c in categories})
        weekly = weekly[weekly_feature_names]
        values = weekly.to_numpy(float)
        profile = profiles.loc[user_id]
        income = float(profile["user_monthly_income"])
        recurring_user = user_expense[user_expense["is_recurring"]]
        recurring_arrays = {}
        for category in RECURRING_CATEGORIES:
            category_rows = recurring_user[recurring_user["category"].eq(category)].sort_values("transaction_timestamp")
            recurring_arrays[category] = (
                pd.DatetimeIndex(category_rows["transaction_timestamp"]),
                category_rows["amount"].to_numpy(float),
            )
        for target_index in range(LOOKBACK, len(weekly)):
            target_start = weekly.index[target_index]
            target_end = target_start + pd.Timedelta(days=6)
            history = values[target_index - LOOKBACK:target_index]
            nonrec_history = history[:, weekly_feature_names.index("nonrec_spending")]
            total_history = history[:, weekly_feature_names.index("total_spending")]
            dynamic_recurring = 0.0
            recurring_context = []
            for category in RECURRING_CATEGORIES:
                category_times, category_amounts = recurring_arrays[category]
                cutoff = int(category_times.searchsorted(target_start, side="left"))
                features = next_due_features(
                    category_times[:cutoff], category_amounts[:cutoff], target_start, target_end, income
                )
                recurring_context.extend(features)
                dynamic_recurring += features[0] * features[1] * income * max(0.35, features[2])
            discretionary_baseline = float(np.median(nonrec_history[-4:]))
            dynamic_baseline = dynamic_recurring + discretionary_baseline
            month_angle = 2 * math.pi * (target_start.month - 1) / 12
            week_angle = 2 * math.pi * (target_start.isocalendar().week - 1) / 52
            dates = pd.date_range(target_start, target_end, freq="D")
            payday = int(profile["user_income_day"])
            prepayday_days = float(sum(((payday - int(day.day)) % 30) <= 3 for day in dates))
            status = str(profile["user_employment_status"])
            context = [
                income, float(profile["user_age"]), float(payday),
                float(status == "student"), float(status == "employed"), float(status == "self_employed"),
                prepayday_days, math.sin(month_angle), math.cos(month_angle),
                math.sin(week_angle), math.cos(week_angle),
                float(np.mean(total_history[-4:])), float(np.median(total_history[-4:])),
                float(np.std(total_history[-4:])), float(np.mean(nonrec_history[-4:])),
                float(np.median(nonrec_history[-4:])), float(dynamic_baseline),
                *recurring_context,
            ]
            rows.append({
                "source": source, "user": str(user_id), "date": target_start,
                "X": np.concatenate([history.reshape(-1), np.asarray(context, dtype=float)]),
                "y": float(values[target_index, weekly_feature_names.index("total_spending")]),
                "baseline": dynamic_baseline, "income": income,
            })
    return {
        "X": np.stack([row["X"] for row in rows]),
        "y": np.asarray([row["y"] for row in rows]),
        "baseline": np.asarray([row["baseline"] for row in rows]),
        "income": np.asarray([row["income"] for row in rows]),
        "date": pd.to_datetime([row["date"] for row in rows]).to_numpy(),
        "user": np.asarray([row["user"] for row in rows]),
        "source": np.asarray([row["source"] for row in rows]),
        "weekly_feature_names": weekly_feature_names,
    }


def metrics(actual: np.ndarray, prediction: np.ndarray) -> dict:
    prediction = np.maximum(0.0, np.asarray(prediction))
    error = prediction - actual
    return {
        "mae": float(np.abs(error).mean()),
        "rmse": float(mean_squared_error(actual, prediction) ** 0.5),
        "wape": float(100 * np.abs(error).sum() / np.abs(actual).sum()),
        "r2": float(r2_score(actual, prediction)),
        "bias": float(error.mean()),
    }

In [ ]:
started = time.perf_counter()
original_forecast = build_samples(original, "original")
stress_forecast = build_samples(stress, "stress_development")
assert original_forecast["X"].shape[1] == stress_forecast["X"].shape[1]
stress_target_dates = pd.to_datetime(stress_forecast["date"])
stress_train_mask = stress_target_dates < pd.Timestamp("2025-09-01")
stress_validation_mask = (
    (stress_target_dates >= pd.Timestamp("2025-09-01"))
    & (stress_target_dates < pd.Timestamp("2025-11-01"))
)
stress_development_eval_mask = stress_target_dates >= pd.Timestamp("2025-11-01")

forecast_x_train = np.concatenate([
    original_forecast["X"], stress_forecast["X"][stress_train_mask]
])
forecast_y_train = np.concatenate([
    original_forecast["y"], stress_forecast["y"][stress_train_mask]
])
forecast_income_train = np.concatenate([
    original_forecast["income"], stress_forecast["income"][stress_train_mask]
])
forecast_x_validation = stress_forecast["X"][stress_validation_mask]
forecast_y_validation = stress_forecast["y"][stress_validation_mask]
forecast_income_validation = stress_forecast["income"][stress_validation_mask]
forecast_x_development_eval = stress_forecast["X"][stress_development_eval_mask]
forecast_y_development_eval = stress_forecast["y"][stress_development_eval_mask]
forecast_income_development_eval = stress_forecast["income"][stress_development_eval_mask]
print({
    "original_samples": len(original_forecast["y"]),
    "stress_samples": len(stress_forecast["y"]),
    "train_samples": len(forecast_y_train),
    "validation_samples": len(forecast_y_validation),
    "development_evaluation_samples": len(forecast_y_development_eval),
    "features": forecast_x_train.shape[1],
    "feature_seconds": time.perf_counter() - started,
})

In [ ]:
forecast_candidates = {
    "hist_absolute_ratio_leaf15": HistGradientBoostingRegressor(
        loss="absolute_error", learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
        min_samples_leaf=30, l2_regularization=10.0, random_state=SEED,
    ),
    "hist_absolute_ratio_leaf31": HistGradientBoostingRegressor(
        loss="absolute_error", learning_rate=0.04, max_iter=400, max_leaf_nodes=31,
        min_samples_leaf=20, l2_regularization=8.0, random_state=SEED,
    ),
    "hist_absolute_ratio_leaf63": HistGradientBoostingRegressor(
        loss="absolute_error", learning_rate=0.035, max_iter=450, max_leaf_nodes=63,
        min_samples_leaf=18, l2_regularization=12.0, random_state=SEED,
    ),
}
forecast_candidate_rows = []
forecast_candidate_predictions = {}
fitted_forecast_candidates = {}
ratio_target = forecast_y_train / np.maximum(forecast_income_train, 1.0)
for name, candidate in forecast_candidates.items():
    model = clone(candidate)
    model.fit(forecast_x_train, ratio_target)
    validation_prediction = np.maximum(
        0.0, model.predict(forecast_x_validation) * forecast_income_validation
    )
    development_prediction = np.maximum(
        0.0, model.predict(forecast_x_development_eval) * forecast_income_development_eval
    )
    validation_metrics = metrics(forecast_y_validation, validation_prediction)
    development_metrics = metrics(forecast_y_development_eval, development_prediction)
    forecast_candidate_rows.append({
        "model": name,
        **{"validation_" + key: value for key, value in validation_metrics.items()},
        **{"development_" + key: value for key, value in development_metrics.items()},
        "mean_wape": (validation_metrics["wape"] + development_metrics["wape"]) / 2,
        "worst_wape": max(validation_metrics["wape"], development_metrics["wape"]),
    })
    forecast_candidate_predictions[name] = development_prediction
    fitted_forecast_candidates[name] = model

forecast_candidate_summary = pd.DataFrame(forecast_candidate_rows).sort_values(
    ["worst_wape", "mean_wape"]
).reset_index(drop=True)
frozen_forecast_name = str(forecast_candidate_summary.iloc[0]["model"])
frozen_forecast_model = fitted_forecast_candidates[frozen_forecast_name]
frozen_forecast_prediction = forecast_candidate_predictions[frozen_forecast_name]
frozen_forecast_development_metrics = metrics(
    forecast_y_development_eval, frozen_forecast_prediction
)
display(forecast_candidate_summary.round(4))
print("Frozen V2 forecast for the new-seed test:", frozen_forecast_name)
print("Development-period metrics:", frozen_forecast_development_metrics)

forecast_development_predictions = pd.DataFrame({
    "user_id": stress_forecast["user"][stress_development_eval_mask],
    "target_week": pd.to_datetime(stress_forecast["date"][stress_development_eval_mask]),
    "actual": forecast_y_development_eval,
    "prediction": frozen_forecast_prediction,
})
forecast_development_predictions["absolute_error"] = (
    forecast_development_predictions["prediction"] - forecast_development_predictions["actual"]
).abs()
forecast_development_predictions["month"] = (
    forecast_development_predictions["target_week"].dt.to_period("M").astype(str)
)
forecast_monthly_metrics = []
for month, part in forecast_development_predictions.groupby("month"):
    forecast_monthly_metrics.append({
        "month": month,
        **metrics(part["actual"].to_numpy(), part["prediction"].to_numpy()),
        "samples": len(part),
    })
forecast_monthly_metrics = pd.DataFrame(forecast_monthly_metrics)
display(forecast_monthly_metrics.round(3))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_table = forecast_candidate_summary.melt(
    id_vars="model", value_vars=["validation_wape", "development_wape"],
    var_name="period", value_name="wape",
)
sns.barplot(data=plot_table, x="wape", y="model", hue="period", ax=axes[0])
axes[0].axvline(PRODUCTION_TARGETS["forecast_wape_max"], linestyle="--", color="#dc2626")
axes[0].set_title("V2 forecast WAPE across development periods")
points = forecast_development_predictions.sample(
    min(2000, len(forecast_development_predictions)), random_state=SEED
)
axes[1].scatter(points["actual"], points["prediction"], alpha=0.25, s=12)
limit = max(points["actual"].max(), points["prediction"].max())
axes[1].plot([0, limit], [0, limit], "--", color="black")
axes[1].set(title="Frozen V2 forecast: actual vs predicted", xlabel="Actual KES", ylabel="Predicted KES")
plt.tight_layout()
plt.show()

## 2. Recurring-aware anomaly detection V2

Normal recurring categories no longer enter the rare-category rule. The model also adds
distinct merchants within one hour, same-merchant/category aggregation, past nighttime rate,
and recurring amount deviation. Stress labels tune thresholds only inside the stated
development periods; no future clean-seed labels are available here.

In [ ]:
def safe_ratio(numerator: float, denominator: float, neutral: float = 0.0) -> float:
    return float(numerator / denominator) if denominator and np.isfinite(denominator) else neutral


def build_past_only_features(raw: pd.DataFrame) -> pd.DataFrame:
    expense = raw[raw["transaction_type"].astype(str).str.lower().eq("expense")].copy()
    expense["transaction_timestamp"] = pd.to_datetime(expense["transaction_timestamp"])
    expense["amount"] = pd.to_numeric(expense["amount"], errors="coerce")
    expense["is_recurring"] = bools(expense["is_recurring"])
    expense["is_anomaly_target"] = bools(expense["is_anomaly"])
    expense = expense.sort_values(["user_id", "transaction_timestamp", "transaction_id"]).reset_index(drop=True)

    engineered_rows: list[dict] = []
    for user_id, group in expense.groupby("user_id", sort=False):
        q1h: deque[tuple[pd.Timestamp, float]] = deque()
        q24h: deque[tuple[pd.Timestamp, float]] = deque()
        q7d: deque[tuple[pd.Timestamp, float]] = deque()
        sum24h = 0.0
        sum7d = 0.0
        user_amounts: list[float] = []
        category_amounts: dict[str, list[float]] = defaultdict(list)
        recurring_amounts: dict[str, list[float]] = defaultdict(list)
        category_counts: dict[str, int] = defaultdict(int)
        last_exact: dict[tuple[str, str, float], pd.Timestamp] = {}
        last_merchant_category: dict[tuple[str, str], pd.Timestamp] = {}
        last_timestamp: pd.Timestamp | None = None
        day_key = None
        day_count = 0
        day_sum = 0.0

        for row in group.itertuples(index=False):
            timestamp = pd.Timestamp(row.transaction_timestamp)
            amount = float(row.amount)
            category = str(row.category)
            merchant = str(row.merchant)
            income = float(row.user_monthly_income)
            current_day = timestamp.date()
            if day_key != current_day:
                day_key = current_day
                day_count = 0
                day_sum = 0.0

            cut1h = timestamp - pd.Timedelta(hours=1)
            cut24h = timestamp - pd.Timedelta(days=1)
            cut7d = timestamp - pd.Timedelta(days=7)
            while q1h and q1h[0][0] < cut1h:
                q1h.popleft()
            while q24h and q24h[0][0] < cut24h:
                sum24h -= q24h.popleft()[1]
            while q7d and q7d[0][0] < cut7d:
                sum7d -= q7d.popleft()[1]

            user_mean = float(np.mean(user_amounts)) if user_amounts else np.nan
            user_median = float(np.median(user_amounts)) if user_amounts else np.nan
            cat_values = category_amounts[category]
            category_mean = float(np.mean(cat_values)) if cat_values else np.nan
            category_median = float(np.median(cat_values)) if cat_values else np.nan
            recurring_values = recurring_amounts[category]
            recurring_median = float(np.median(recurring_values)) if recurring_values else np.nan
            exact_key = (merchant, category, round(amount, 2))
            merchant_key = (merchant, category)
            exact_age = (
                (timestamp - last_exact[exact_key]).total_seconds() / 60.0
                if exact_key in last_exact else np.nan
            )
            merchant_age = (
                (timestamp - last_merchant_category[merchant_key]).total_seconds() / 60.0
                if merchant_key in last_merchant_category else np.nan
            )
            past_count = len(user_amounts)

            engineered_rows.append({
                "transaction_id": row.transaction_id,
                "log_amount": np.log1p(max(0.0, amount)),
                "amount_to_income": safe_ratio(amount, income),
                "amount_to_user_mean": safe_ratio(amount, user_mean, 1.0),
                "amount_to_user_median": safe_ratio(amount, user_median, 1.0),
                "amount_to_category_mean": safe_ratio(amount, category_mean, 1.0),
                "amount_to_category_median": safe_ratio(amount, category_median, 1.0),
                "recurring_amount_ratio": safe_ratio(amount, recurring_median, 1.0),
                "past_user_count": past_count,
                "category_proportion": safe_ratio(category_counts[category], past_count),
                "previous_1h_count": len(q1h),
                "previous_24h_count": len(q24h),
                "previous_7d_count": len(q7d),
                "previous_24h_sum_to_income": safe_ratio(sum24h, income),
                "previous_7d_sum_to_income": safe_ratio(sum7d, income),
                "same_day_count_before": day_count,
                "same_day_sum_to_income": safe_ratio(day_sum, income),
                "time_since_previous_hours": (
                    (timestamp - last_timestamp).total_seconds() / 3600.0 if last_timestamp is not None else np.nan
                ),
                "same_exact_age_minutes": exact_age,
                "same_merchant_category_age_minutes": merchant_age,
                "is_recurring_feature": int(bool(row.is_recurring)),
                "is_weekend": int(timestamp.dayofweek >= 5),
                "hour_sin": np.sin(2 * np.pi * timestamp.hour / 24),
                "hour_cos": np.cos(2 * np.pi * timestamp.hour / 24),
            })

            q1h.append((timestamp, amount))
            q24h.append((timestamp, amount))
            q7d.append((timestamp, amount))
            sum24h += amount
            sum7d += amount
            user_amounts.append(amount)
            category_amounts[category].append(amount)
            category_counts[category] += 1
            if bool(row.is_recurring):
                recurring_amounts[category].append(amount)
            last_exact[exact_key] = timestamp
            last_merchant_category[merchant_key] = timestamp
            last_timestamp = timestamp
            day_count += 1
            day_sum += amount

    engineered = pd.DataFrame(engineered_rows)
    return expense.merge(engineered, on="transaction_id", how="left", validate="one_to_one")


def split_chronologically(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    unique_times = np.sort(frame["transaction_timestamp"].unique())
    train_count = max(1, int(np.floor(len(unique_times) * 0.70)))
    train_end = unique_times[train_count - 1]
    train = frame[frame["transaction_timestamp"] <= train_end].copy()
    held = frame[frame["transaction_timestamp"] > train_end].copy()
    held_times = np.sort(held["transaction_timestamp"].unique())
    val_count = max(1, int(np.floor(len(held_times) * 0.20)))
    val_end = held_times[val_count - 1]
    validation = held[held["transaction_timestamp"] <= val_end].copy()
    evaluation = held[held["transaction_timestamp"] > val_end].copy()
    return train, validation, evaluation


def anomaly_metric_record(actual: np.ndarray, predicted: np.ndarray) -> dict:
    actual = np.asarray(actual, dtype=bool)
    predicted = np.asarray(predicted, dtype=bool)
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[False, True]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        actual, predicted, average="binary", zero_division=0
    )
    return {
        "precision": float(precision), "recall": float(recall), "f1": float(f1),
        "false_positive_rate": float(fp / (fp + tn)), "tp": int(tp), "fp": int(fp),
        "tn": int(tn), "fn": int(fn), "alerts": int(predicted.sum()),
    }


def best_threshold(labels: np.ndarray, scores: np.ndarray) -> tuple[float, dict]:
    precision, recall, thresholds = precision_recall_curve(labels, scores)
    if not len(thresholds):
        threshold = float(np.max(scores) + 1)
    else:
        f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
        threshold = float(thresholds[int(np.nanargmax(f1))])
    return threshold, anomaly_metric_record(labels, scores >= threshold)


def rules(frame: pd.DataFrame, config: dict) -> tuple[np.ndarray, np.ndarray]:
    duplicate = frame["same_exact_age_minutes"].between(0, config["duplicate_minutes"], inclusive="both")
    burst = frame["previous_1h_count"].ge(config["burst_count"])
    recurring = (
        frame["is_recurring_feature"].eq(1)
        & frame["category"].eq("Rent")
        & frame["recurring_amount_ratio"].ge(config["recurring_ratio"])
        & frame["past_user_count"].ge(20)
    )
    weekend = (
        frame["is_weekend"].eq(1)
        & frame["category"].eq("Entertainment")
        & frame["amount_to_income"].ge(config["weekend_ratio"])
    )
    random_shock = (
        frame["category"].eq("Irregular Expenses")
        & frame["amount_to_income"].ge(config["shock_ratio"])
    )
    extreme = frame["amount_to_income"].ge(config["extreme_ratio"])
    rare = (
        frame["past_user_count"].ge(30)
        & frame["category_proportion"].le(config["rare_proportion"])
        & frame["amount_to_income"].ge(config["rare_amount_ratio"])
    )
    flags = np.column_stack([duplicate, burst, recurring, weekend, random_shock, extreme, rare]).astype(int)
    weights = np.array([3.0, 2.0, 3.0, 2.0, 3.0, 3.0, 2.0])
    return flags.dot(weights), flags


def percentile_against(reference: np.ndarray, values: np.ndarray) -> np.ndarray:
    ordered = np.sort(np.asarray(reference, dtype=float))
    return np.searchsorted(ordered, values, side="right") / max(1, len(ordered))


def add_cluster_features(frame: pd.DataFrame) -> pd.DataFrame:
    outputs = []
    for _, group in frame.groupby("user_id", sort=False):
        queue = deque()
        past_count = 0
        past_night_count = 0
        for row in group.sort_values(["transaction_timestamp", "transaction_id"]).itertuples(index=False):
            timestamp = pd.Timestamp(row.transaction_timestamp)
            cutoff = timestamp - pd.Timedelta(hours=1)
            while queue and queue[0][0] < cutoff:
                queue.popleft()
            same = [item for item in queue if item[2] == str(row.merchant) and item[3] == str(row.category)]
            income = max(1.0, float(row.user_monthly_income))
            outputs.append({
                "transaction_id": row.transaction_id,
                "distinct_merchants_1h": float(len({item[2] for item in queue})),
                "same_merchant_category_count_1h": float(len(same)),
                "total_amount_with_current_1h_to_income": float((sum(item[1] for item in queue) + float(row.amount)) / income),
                "same_merchant_category_sum_with_current_1h_to_income": float((sum(item[1] for item in same) + float(row.amount)) / income),
                "historical_night_proportion": float(past_night_count / past_count) if past_count else 0.0,
                "is_night": int(timestamp.hour <= 5),
                "is_known_recurring_category": int(str(row.category) in KNOWN_RECURRING),
            })
            queue.append((timestamp, float(row.amount), str(row.merchant), str(row.category)))
            past_count += 1
            past_night_count += int(timestamp.hour <= 5)
    return frame.merge(pd.DataFrame(outputs), on="transaction_id", how="left", validate="one_to_one")


def v2_rule_scores(frame: pd.DataFrame, config: dict) -> tuple[np.ndarray, np.ndarray]:
    nonrecurring = frame["is_recurring_feature"].eq(0)
    duplicate = frame["same_exact_age_minutes"].between(0, config["duplicate_minutes"], inclusive="both")
    burst = (
        frame["previous_1h_count"].ge(config["burst_count"])
        & frame["total_amount_with_current_1h_to_income"].ge(config["burst_ratio"])
    )
    recurring_increase = (
        frame["is_recurring_feature"].eq(1)
        & frame["is_known_recurring_category"].eq(1)
        & frame["recurring_amount_ratio"].ge(config["recurring_ratio"])
        & frame["past_user_count"].ge(20)
    )
    weekend = (
        frame["is_weekend"].eq(1)
        & frame["category"].eq("Entertainment")
        & frame["amount_to_income"].ge(config["weekend_ratio"])
    )
    random_shock = (
        frame["category"].eq("Irregular Expenses")
        & frame["amount_to_income"].ge(config["shock_ratio"])
    )
    extreme = nonrecurring & frame["amount_to_income"].ge(config["extreme_ratio"])
    rare = (
        nonrecurring
        & frame["is_known_recurring_category"].eq(0)
        & frame["past_user_count"].ge(30)
        & frame["category_proportion"].le(config["rare_proportion"])
        & frame["amount_to_income"].ge(config["rare_amount_ratio"])
    )
    merchant_hopping = (
        frame["previous_1h_count"].ge(2)
        & frame["distinct_merchants_1h"].ge(config["distinct_merchants"])
        & frame["total_amount_with_current_1h_to_income"].ge(config["merchant_hopping_ratio"])
    )
    split_pattern = (
        frame["same_merchant_category_count_1h"].ge(2)
        & frame["same_merchant_category_sum_with_current_1h_to_income"].ge(config["split_ratio"])
    )
    nighttime = (
        frame["is_night"].eq(1)
        & frame["historical_night_proportion"].le(config["night_history_max"])
        & frame["amount_to_income"].ge(config["night_ratio"])
    )
    flags = np.column_stack([
        duplicate, burst, recurring_increase, weekend, random_shock,
        extreme, rare, merchant_hopping, split_pattern, nighttime,
    ]).astype(int)
    weights = np.array([3.0, 2.0, 3.0, 2.0, 3.0, 3.0, 2.0, 2.5, 3.0, 3.0])
    return flags.dot(weights), flags

In [ ]:
started = time.perf_counter()
original_anomaly_frame = add_cluster_features(build_past_only_features(original))
stress_anomaly_frame = add_cluster_features(build_past_only_features(stress))
original_anomaly_train, _, _ = split_chronologically(original_anomaly_frame)
stress_times = stress_anomaly_frame["transaction_timestamp"]
stress_anomaly_train = stress_anomaly_frame[stress_times < pd.Timestamp("2025-09-01")].copy()
stress_anomaly_validation = stress_anomaly_frame[
    (stress_times >= pd.Timestamp("2025-09-01")) & (stress_times < pd.Timestamp("2025-11-01"))
].copy()
stress_anomaly_development_eval = stress_anomaly_frame[
    stress_times >= pd.Timestamp("2025-11-01")
].copy()

anomaly_features = [
    "log_amount", "amount_to_income", "amount_to_user_mean", "amount_to_user_median",
    "amount_to_category_mean", "amount_to_category_median", "recurring_amount_ratio",
    "category_proportion", "previous_1h_count", "previous_24h_count", "previous_7d_count",
    "previous_24h_sum_to_income", "previous_7d_sum_to_income", "same_day_count_before",
    "same_day_sum_to_income", "time_since_previous_hours", "same_exact_age_minutes",
    "same_merchant_category_age_minutes", "is_recurring_feature", "is_weekend", "hour_sin", "hour_cos",
    "distinct_merchants_1h", "same_merchant_category_count_1h",
    "total_amount_with_current_1h_to_income", "same_merchant_category_sum_with_current_1h_to_income",
    "historical_night_proportion", "is_night", "is_known_recurring_category",
]
combined_anomaly_train = pd.concat(
    [original_anomaly_train, stress_anomaly_train], ignore_index=True
)
for part in [
    combined_anomaly_train, stress_anomaly_validation, stress_anomaly_development_eval
]:
    part[anomaly_features] = part[anomaly_features].replace([np.inf, -np.inf], np.nan)
anomaly_fill_values = combined_anomaly_train[anomaly_features].median().fillna(0.0)
v2_anomaly_scaler = RobustScaler(quantile_range=(10, 90))
anomaly_x_train = v2_anomaly_scaler.fit_transform(
    combined_anomaly_train[anomaly_features].fillna(anomaly_fill_values)
)
anomaly_x_validation = v2_anomaly_scaler.transform(
    stress_anomaly_validation[anomaly_features].fillna(anomaly_fill_values)
)
anomaly_x_development_eval = v2_anomaly_scaler.transform(
    stress_anomaly_development_eval[anomaly_features].fillna(anomaly_fill_values)
)
anomaly_y_validation = stress_anomaly_validation["is_anomaly_target"].to_numpy(bool)
anomaly_y_development_eval = stress_anomaly_development_eval["is_anomaly_target"].to_numpy(bool)
print({
    "train_rows": len(combined_anomaly_train),
    "validation_rows": len(stress_anomaly_validation),
    "development_evaluation_rows": len(stress_anomaly_development_eval),
    "validation_anomalies": int(anomaly_y_validation.sum()),
    "development_evaluation_anomalies": int(anomaly_y_development_eval.sum()),
    "feature_seconds": time.perf_counter() - started,
})

In [ ]:
if_candidates = []
if_models = {}
for max_samples in [256, 1024, 4096]:
    model = IsolationForest(
        n_estimators=400, max_samples=max_samples, max_features=1.0,
        contamination="auto", random_state=SEED, n_jobs=-1,
    ).fit(anomaly_x_train)
    validation_scores = -model.score_samples(anomaly_x_validation)
    threshold, record = best_threshold(anomaly_y_validation, validation_scores)
    name = f"if_{max_samples}"
    if_candidates.append({"model": name, "threshold": threshold, **record})
    if_models[name] = (model, threshold, validation_scores)
if_validation = pd.DataFrame(if_candidates).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
frozen_if_name = str(if_validation.iloc[0]["model"])
frozen_v2_if, frozen_if_threshold, validation_if_scores = if_models[frozen_if_name]
training_if_scores = -frozen_v2_if.score_samples(anomaly_x_train)
development_if_scores = -frozen_v2_if.score_samples(anomaly_x_development_eval)
display(if_validation.round(4))

v2_rule_configs = []
for burst_count in [2, 3]:
    for burst_ratio in [0.015, 0.030]:
        for night_ratio in [0.04, 0.06]:
            v2_rule_configs.append({
                "duplicate_minutes": 5, "burst_count": burst_count, "burst_ratio": burst_ratio,
                "recurring_ratio": 1.40, "weekend_ratio": 0.06, "shock_ratio": 0.15,
                "extreme_ratio": 0.30, "rare_proportion": 0.025, "rare_amount_ratio": 0.15,
                "distinct_merchants": 2, "merchant_hopping_ratio": 0.025,
                "split_ratio": 0.14, "night_history_max": 0.02,
                "night_ratio": night_ratio,
            })
v2_rule_rows = []
for config_index, config in enumerate(v2_rule_configs):
    validation_rule_score, _ = v2_rule_scores(stress_anomaly_validation, config)
    for threshold in [2.0, 2.5, 3.0, 3.5, 4.0]:
        v2_rule_rows.append({
            "config_index": config_index, "threshold": threshold,
            **anomaly_metric_record(
                anomaly_y_validation, validation_rule_score >= threshold
            ),
        })
v2_rule_validation = pd.DataFrame(v2_rule_rows).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
best_rule_row = v2_rule_validation.iloc[0]
frozen_v2_rule_config = v2_rule_configs[int(best_rule_row["config_index"])]
frozen_v2_rule_threshold = float(best_rule_row["threshold"])
validation_rule_scores, _ = v2_rule_scores(
    stress_anomaly_validation, frozen_v2_rule_config
)
development_rule_scores, development_rule_flags = v2_rule_scores(
    stress_anomaly_development_eval, frozen_v2_rule_config
)

validation_if_percentile = percentile_against(training_if_scores, validation_if_scores)
development_if_percentile = percentile_against(training_if_scores, development_if_scores)
hybrid_rows = []
for if_weight in [0.0, 0.25, 0.50, 0.75, 1.0]:
    validation_hybrid_score = validation_rule_scores + if_weight * validation_if_percentile
    threshold, record = best_threshold(anomaly_y_validation, validation_hybrid_score)
    hybrid_rows.append({"if_weight": if_weight, "threshold": threshold, **record})
hybrid_validation = pd.DataFrame(hybrid_rows).sort_values(
    ["f1", "precision"], ascending=False
).reset_index(drop=True)
frozen_hybrid = hybrid_validation.iloc[0]
development_hybrid_score = (
    development_rule_scores
    + float(frozen_hybrid["if_weight"]) * development_if_percentile
)
development_anomaly_prediction = (
    development_hybrid_score >= float(frozen_hybrid["threshold"])
)
v2_anomaly_development_metrics = anomaly_metric_record(
    anomaly_y_development_eval, development_anomaly_prediction
)
print("Best recurring-aware rule configuration")
display(pd.Series({**frozen_v2_rule_config, **best_rule_row.to_dict()}, name="value").to_frame())
display(hybrid_validation.round(4))
display(pd.Series(v2_anomaly_development_metrics, name="development evaluation").to_frame())

In [ ]:
v2_anomaly_predictions = stress_anomaly_development_eval[[
    "transaction_id", "user_id", "transaction_timestamp", "amount", "category",
    "merchant", "anomaly_type", "is_anomaly_target",
]].copy()
v2_anomaly_predictions["hybrid_score"] = development_hybrid_score
v2_anomaly_predictions["predicted_anomaly"] = development_anomaly_prediction
v2_anomaly_scenarios = (
    v2_anomaly_predictions[v2_anomaly_predictions["is_anomaly_target"]]
    .groupby("anomaly_type")["predicted_anomaly"].agg(["size", "sum"])
    .reset_index().rename(columns={"size": "labelled", "sum": "detected"})
)
v2_anomaly_scenarios["recall"] = (
    v2_anomaly_scenarios["detected"] / v2_anomaly_scenarios["labelled"]
)
v2_false_positive_categories = (
    v2_anomaly_predictions[
        (~v2_anomaly_predictions["is_anomaly_target"])
        & v2_anomaly_predictions["predicted_anomaly"]
    ]["category"].value_counts().rename_axis("category").reset_index(name="false_positives")
)
display(v2_anomaly_scenarios.round(4))
display(v2_false_positive_categories)

matrix = confusion_matrix(
    anomaly_y_development_eval, development_anomaly_prediction, labels=[False, True]
)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(data=v2_anomaly_scenarios.sort_values("recall"), x="recall", y="anomaly_type", ax=axes[0], color="#16a34a")
axes[0].axvline(PRODUCTION_TARGETS["anomaly_recall_min"], linestyle="--", color="#dc2626")
axes[0].set_xlim(0, 1)
axes[0].set_title("V2 anomaly recall by scenario")
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[1],
            xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
axes[1].set(title="V2 development confusion matrix", xlabel="Predicted", ylabel="Controlled label")
plt.tight_layout()
plt.show()

## 3. Freeze V2 and export development evidence

Passing development metrics do not authorize production integration. The saved artifacts are
candidates for one clean run on a different seed, with no further threshold changes.

In [ ]:
forecast_development_checks = {
    "wape_at_or_below_target": bool(
        frozen_forecast_development_metrics["wape"] <= PRODUCTION_TARGETS["forecast_wape_max"]
    ),
    "r2_at_or_above_target": bool(
        frozen_forecast_development_metrics["r2"] >= PRODUCTION_TARGETS["forecast_r2_min"]
    ),
}
anomaly_development_checks = {
    "precision_at_or_above_target": bool(
        v2_anomaly_development_metrics["precision"] >= PRODUCTION_TARGETS["anomaly_precision_min"]
    ),
    "recall_at_or_above_target": bool(
        v2_anomaly_development_metrics["recall"] >= PRODUCTION_TARGETS["anomaly_recall_min"]
    ),
}
ready_for_final_seed_test = (
    forecast_development_checks["r2_at_or_above_target"]
    and frozen_forecast_development_metrics["wape"] <= 32.0
    and all(anomaly_development_checks.values())
)
next_action = (
    "Run the frozen V2 configuration once on a newly generated untouched seed."
    if ready_for_final_seed_test
    else "Continue bounded development before creating the final untouched seed."
)

forecast_candidate_summary.to_csv(OUTPUT_DIR / "forecast_v2_candidate_metrics.csv", index=False)
forecast_development_predictions.to_csv(OUTPUT_DIR / "forecast_v2_development_predictions.csv", index=False)
forecast_monthly_metrics.to_csv(OUTPUT_DIR / "forecast_v2_monthly_metrics.csv", index=False)
if_validation.to_csv(OUTPUT_DIR / "anomaly_v2_if_validation.csv", index=False)
v2_rule_validation.to_csv(OUTPUT_DIR / "anomaly_v2_rule_validation.csv", index=False)
hybrid_validation.to_csv(OUTPUT_DIR / "anomaly_v2_hybrid_validation.csv", index=False)
v2_anomaly_predictions.to_csv(OUTPUT_DIR / "anomaly_v2_development_predictions.csv", index=False)
v2_anomaly_scenarios.to_csv(OUTPUT_DIR / "anomaly_v2_scenario_metrics.csv", index=False)
v2_false_positive_categories.to_csv(OUTPUT_DIR / "anomaly_v2_false_positive_categories.csv", index=False)
joblib.dump(frozen_forecast_model, OUTPUT_DIR / "frozen_v2_forecast.joblib")
joblib.dump(frozen_v2_if, OUTPUT_DIR / "frozen_v2_isolation_forest.joblib")
joblib.dump(v2_anomaly_scaler, OUTPUT_DIR / "frozen_v2_anomaly_scaler.joblib")

feature_schema = {
    "forecast_feature_count": int(forecast_x_train.shape[1]),
    "forecast_weekly_features": original_forecast["weekly_feature_names"],
    "forecast_lookback_weeks": LOOKBACK,
    "forecast_target_transform": "weekly_spending_divided_by_monthly_income",
    "anomaly_features": anomaly_features,
    "anomaly_fill_values": anomaly_fill_values.to_dict(),
    "anomaly_rule_config": frozen_v2_rule_config,
    "anomaly_rule_threshold": frozen_v2_rule_threshold,
    "anomaly_if_weight": float(frozen_hybrid["if_weight"]),
    "anomaly_hybrid_threshold": float(frozen_hybrid["threshold"]),
}
(OUTPUT_DIR / "v2_feature_schema.json").write_text(
    json.dumps(feature_schema, indent=2, default=float), encoding="utf-8"
)

result_payload = {
    "notebook": "Spending_Model_Development_V2",
    "original_dataset_sha256": original_sha256,
    "development_stress_rows": int(len(stress)),
    "development_stress_users": int(stress["user_id"].nunique()),
    "frozen_forecast_model": frozen_forecast_name,
    "forecast_candidate_metrics": forecast_candidate_summary.round(6).to_dict(orient="records"),
    "frozen_forecast_development_metrics": frozen_forecast_development_metrics,
    "forecast_development_checks": forecast_development_checks,
    "frozen_anomaly_if": frozen_if_name,
    "anomaly_validation_metrics": hybrid_validation.iloc[0].to_dict(),
    "anomaly_development_metrics": v2_anomaly_development_metrics,
    "anomaly_development_checks": anomaly_development_checks,
    "scenario_metrics": v2_anomaly_scenarios.round(6).to_dict(orient="records"),
    "ready_for_final_seed_test": bool(ready_for_final_seed_test),
    "next_action": next_action,
    "checks": {
        "full_weeks_only": True,
        "past_only_schedule_inference": True,
        "recurring_categories_excluded_from_rare_rule": True,
        "current_stress_seed_treated_as_development": True,
        "new_final_seed_opened": False,
        "app_changed": False,
    },
}
results_path = OUTPUT_DIR / "spendly_development_v2_results.json"
results_path.write_text(json.dumps(result_payload, indent=2, default=str), encoding="utf-8")
zip_path = Path("spendly_development_v2_output.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT_DIR.parent))
print("SPENDLY_DEVELOPMENT_V2_RESULT_START")
print(json.dumps(result_payload, indent=2, default=str))
print("SPENDLY_DEVELOPMENT_V2_RESULT_END")
print("Saved:", zip_path.resolve())
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

## Interpretation

`ready_for_final_seed_test` means V2 is sufficiently improved to justify one untouched
generalization test. It does **not** mean the app should be changed yet. App integration begins
only if the frozen V2 models pass the original production targets on that new seed.